# 每月调仓决策助手 v2

**策略**: U12 + residual_mom + n=4 + rank weighting + BTC 1% sleeve

**Universe (12 个 main + 1 个 satellite)**:
- US equity: SPY, QQQ
- 国际: EFA, VWO
- 利率/信用: TLT, IEF, HYG
- 商品/能源: GLD, DBC, **XOM**（替换 USO，避开 contango decay）
- 半导体: SOXX
- REITs: VNQ
- **BTC satellite**: 1% capped sleeve（信号用 BTC-USD，实盘买 IBIT）

**操作流程**（每月第一个交易日做一次）：
1. 在 Inputs cell 里填写**当前持仓** + **可用现金**
2. Cell → Run All
3. 最后一格输出**交易清单**：去 IBKR 下单

**Backtest 表现** (2015-2026, 11.4 年):
- CAGR **17.89%** / Sharpe **1.29** / MDD **-17.0%**
- vs SPY: +3.9pp CAGR、+0.45 Sharpe、MDD 一半

**Forward 期望**（打 50-70% 折扣后）: Sharpe 0.7-0.9, CAGR 10-14%, MDD -20% to -28%

## 1. 配置 + Imports（不改）

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Main universe — feeds the cross-sectional rank ranking
MAIN_UNIVERSE = ['SPY', 'QQQ', 'EFA', 'TLT', 'GLD',
                 'VWO', 'HYG', 'VNQ', 'IEF', 'DBC',
                 'SOXX', 'XOM']
LONG_N = 4                  # 持有 4 个 winners
RANK_WEIGHTS_N4 = [0.40, 0.30, 0.20, 0.10]   # rank weighting top→bottom

# BTC satellite sleeve
BTC_SIGNAL_TICKER = 'BTC-USD'   # 用于信号计算（yfinance 实时 spot）
BTC_TRADE_TICKER  = 'IBIT'      # 实盘买的 ETF（iShares Bitcoin Trust）
BTC_CAP           = 0.01        # 1% cap

# All tickers to fetch
FETCH_TICKERS = MAIN_UNIVERSE + [BTC_SIGNAL_TICKER, BTC_TRADE_TICKER]

print(f'Strategy: U{len(MAIN_UNIVERSE)} + residual_mom + n={LONG_N} + rank + BTC {BTC_CAP*100:.0f}%')
print(f'Main universe: {MAIN_UNIVERSE}')
print(f'BTC sleeve: signal={BTC_SIGNAL_TICKER}, trade={BTC_TRADE_TICKER}, cap={BTC_CAP*100:.0f}%')

## 2. ★ Inputs ★（**每月只改这一格**）

把 IBKR 账户里**今天**的实际状态填进去。BTC 部分填 IBIT 实际持有股数（不是 BTC-USD 信号 ticker）：

In [ ]:
# 当前持仓：ticker -> 持有股数（整数）。
# 没持有的不要列；首次运行就留空 dict {}
# 注意：BTC 用 IBIT（不是 BTC-USD）
current_holdings = {
    # 'SPY':  12,
    # 'TLT':  25,
    # 'GLD':  18,
    # 'SOXX':  5,
    # 'IBIT': 30,
}

# 当前可用现金余额（USD），包含本月刚 deposit 的钱
cash_balance = 23000.0     # 例：$20k 起始 + $3k 月度 deposit

# As-of date：通常 today。指定历史日期可做回溯演练。
as_of_date = None          # 例如: '2026-06-02'

# ─── derived ───
if as_of_date is None:
    as_of_date = pd.Timestamp.today().normalize()
else:
    as_of_date = pd.Timestamp(as_of_date)
print(f'As of: {as_of_date.date()}')
print(f'Cash:  ${cash_balance:,.2f}')
print(f'Holdings: {dict(current_holdings) or "(empty — first run)"}')

## 3. 拉数据（自动）

In [ ]:
fetch_start = (as_of_date - timedelta(days=400)).strftime('%Y-%m-%d')
fetch_end   = (as_of_date + timedelta(days=1)).strftime('%Y-%m-%d')

raw = yf.download(FETCH_TICKERS, start=fetch_start, end=fetch_end,
                  progress=False, auto_adjust=True)
# Align to SPY trading days (BTC trades 24/7; we use weekday closes only)
closes = raw['Close'].dropna(subset=['SPY']) if 'SPY' in raw['Close'].columns else raw['Close']

print(f'Data fetched: {len(closes)} trading days from '
      f'{closes.index.min().date()} to {closes.index.max().date()}')

latest_prices = closes.iloc[-1]
print(f'\nLatest prices (as of {closes.index[-1].date()}):')
for t in FETCH_TICKERS:
    p = latest_prices.get(t)
    if pd.notna(p):
        print(f'  {t:>10}: ${p:>9.2f}')
    else:
        print(f'  {t:>10}: MISSING')

## 4. 计算 residual_mom 信号

对每个 ticker：剥除其相对 SPY 的 beta，残差累积回报作为信号。BTC 也独立计算。

In [ ]:
def residual_mom_latest(asset_px, market_px):
    asset_ret = asset_px.pct_change()
    mkt_ret = market_px.pct_change()
    aligned = pd.concat([asset_ret, mkt_ret], axis=1, keys=['a', 'm']).dropna()
    if len(aligned) < 252:
        return np.nan
    win = aligned.iloc[-252:-21]
    if len(win) < 100:
        return np.nan
    a, m = win['a'].values, win['m'].values
    cov = float(np.cov(a, m, ddof=0)[0, 1])
    var = float(np.var(m, ddof=0))
    beta = cov / var if var > 1e-12 else 1.0
    return float((a - beta * m).sum())

# Main universe signals
main_signals = {t: residual_mom_latest(closes[t], closes['SPY'])
                for t in MAIN_UNIVERSE if t in closes.columns}

# BTC signal (separate)
btc_signal = residual_mom_latest(closes[BTC_SIGNAL_TICKER], closes['SPY']) \
             if BTC_SIGNAL_TICKER in closes.columns else np.nan

sig_df = pd.DataFrame({
    'signal':     [main_signals.get(t, np.nan) for t in MAIN_UNIVERSE],
    'positive':   [main_signals.get(t, np.nan) > 0 if not pd.isna(main_signals.get(t, np.nan)) else False
                   for t in MAIN_UNIVERSE],
    'last_price': [latest_prices.get(t) for t in MAIN_UNIVERSE],
}, index=MAIN_UNIVERSE).sort_values('signal', ascending=False)
sig_df['rank'] = range(1, len(sig_df) + 1)

print('Main universe signal table:')
with pd.option_context('display.float_format', '{:+.4f}'.format):
    print(sig_df.to_string())

print(f'\nBTC signal: {btc_signal:+.4f} ({"positive — allocate 1%" if not pd.isna(btc_signal) and btc_signal > 0 else "negative or NaN — skip BTC"})')

## 5. 计算目标分配（rank + BTC sleeve）

In [ ]:
# Step 1: BTC sleeve allocation
if not pd.isna(btc_signal) and btc_signal > 0:
    btc_alloc = BTC_CAP   # 1% to BTC
else:
    btc_alloc = 0.0
main_alloc = 1.0 - btc_alloc

# Step 2: Main strategy — top 4 positive momentum, rank weighting
positives = sig_df[sig_df['positive']].head(LONG_N)
k = len(positives)

if k == 0:
    print('⚠️  Main universe 没有正动量 ETF → main 持现金。')
    targets_pct = {}
elif k < LONG_N:
    print(f'⚠️  只有 {k}/{LONG_N} 个 main ETF 正动量。Main 部分留 {(1-k/LONG_N)*100:.0f}% 现金。')
    weights_top4 = [(LONG_N - i) for i in range(LONG_N)]
    sum_all = sum(weights_top4)
    targets_pct = {positives.index[i]: (weights_top4[i] / sum_all) * main_alloc
                    for i in range(k)}
else:
    targets_pct = {positives.index[i]: RANK_WEIGHTS_N4[i] * main_alloc
                    for i in range(LONG_N)}

# Step 3: Add BTC sleeve (using IBIT for execution)
if btc_alloc > 0:
    targets_pct[BTC_TRADE_TICKER] = btc_alloc

# Compute current portfolio value
holdings_value = sum(shares * latest_prices.get(t, 0)
                     for t, shares in current_holdings.items()
                     if t in latest_prices and pd.notna(latest_prices.get(t)))
total_value = cash_balance + holdings_value
print(f'\n当前组合价值：${total_value:,.2f}  (cash ${cash_balance:,.2f} + holdings ${holdings_value:,.2f})')

print(f'\n目标分配 (BTC sleeve = {btc_alloc*100:.1f}%, main = {main_alloc*100:.1f}%):')
for t, w in sorted(targets_pct.items(), key=lambda kv: -kv[1]):
    target_dollars = total_value * w
    price = latest_prices[t]
    target_shares = int(round(target_dollars / price)) if price > 0 else 0
    note = ' [BTC sleeve via IBIT]' if t == BTC_TRADE_TICKER else ''
    print(f'  {t:>6} weight {w*100:>5.2f}%  → ${target_dollars:>10,.0f} ≈ {target_shares:>4d} shares @ ${price:.2f}{note}')

## 6. 交易清单（**复制到 IBKR 执行**）

In [ ]:
all_relevant_tickers = list(set(MAIN_UNIVERSE) | set(targets_pct.keys()) | set(current_holdings.keys()))

target_shares = {}
for t in all_relevant_tickers:
    if t in targets_pct and t in latest_prices and pd.notna(latest_prices[t]) and latest_prices[t] > 0:
        target_shares[t] = int(round(total_value * targets_pct[t] / latest_prices[t]))
    else:
        target_shares[t] = 0

trade_rows = []
for t in all_relevant_tickers:
    curr = current_holdings.get(t, 0)
    tgt = target_shares.get(t, 0)
    diff = tgt - curr
    if diff == 0 and curr == 0:
        continue
    if t not in latest_prices or pd.isna(latest_prices.get(t)):
        continue
    action = 'HOLD' if diff == 0 else ('BUY' if diff > 0 else 'SELL')
    trade_rows.append({
        'action':  action, 'ticker':  t,
        'current': curr,  'target':  tgt, 'delta': diff,
        'price':   latest_prices[t],
        'amount':  abs(diff) * latest_prices[t],
    })

trade_df = pd.DataFrame(trade_rows)
if trade_df.empty:
    print('✅ 当前持仓已与目标一致，本月无需交易。')
else:
    trade_df = trade_df.sort_values(['action', 'amount'], ascending=[True, False])
    sells = trade_df[trade_df.action == 'SELL']
    buys  = trade_df[trade_df.action == 'BUY']
    holds = trade_df[trade_df.action == 'HOLD']

    print('═' * 64)
    print(f'   交易清单（{as_of_date.date()}）')
    print('═' * 64)
    if not sells.empty:
        print('\n→ 先 SELL（先卖出腾现金）:')
        for _, r in sells.iterrows():
            print(f"   SELL  {r['ticker']:>6}  {r['delta']:>+5d} shares  @ ~${r['price']:>8.2f}  = ${r['amount']:>10,.0f}")
    if not buys.empty:
        print('\n→ 再 BUY:')
        for _, r in buys.iterrows():
            print(f"   BUY   {r['ticker']:>6}  {r['delta']:>+5d} shares  @ ~${r['price']:>8.2f}  = ${r['amount']:>10,.0f}")
    if not holds.empty:
        print('\n→ HOLD（不动）:')
        for _, r in holds.iterrows():
            print(f"   HOLD  {r['ticker']:>6}  {r['current']:>5d} shares  (no change)")

    total_sell = sells['amount'].sum() if not sells.empty else 0
    total_buy  = buys['amount'].sum()  if not buys.empty  else 0
    print(f"\n💵 净现金变动: SELL ${total_sell:,.0f} - BUY ${total_buy:,.0f} = ${total_sell - total_buy:+,.0f}")
    print(f"💰 交易完成后预计现金: ${cash_balance + total_sell - total_buy:,.0f}")
    print('═' * 64)

## 7. 健康检查 + 备忘

**执行注意**：
- 用 market order（month rebalance 不在乎 1-2bp 滑点）
- **先卖后买**
- **BTC 用 IBIT 买**，不是 GBTC 不是 FBTC（虽然都跟踪 BTC，IBIT 流动性最好、expense ratio 0.25%）
- 整数股，零碎现金留作下月 buffer

**关于 BTC sleeve**：
- 信号用 BTC-USD（spot）算，实盘买 **IBIT**
- 1% 仓位很小（$20k → $200），即使 BTC 翻 10 倍也只贡献 2k 增值
- 月度调仓自动 cap 在 1%——BTC 涨了卖回，跌了不补
- 如果未来想加大 BTC 暴露，把 `BTC_CAP` 改到 0.02 / 0.05（2% / 5%）

**异常处理**：
- 全资产负动量（罕见）→ Main 全持现金，BTC 也不要（按规则）
- 某月 turnover > 60% → 检查信号计算是否异常
- 实盘 Sharpe 长期 < 0.5 → 策略 forward 性能不达预期，review or pause

**调仓时间**：每月第一个交易日 10:00 ET 后跑（确保前日数据已 settle）。

**Deposit 节奏**：下次 $3k 到账后更新 `cash_balance`，notebook 自动把新现金 deploy。

In [ ]:
# Save decision log for audit trail
log_path = f'rebalance_log_{as_of_date.date()}.txt'
with open(log_path, 'w') as f:
    f.write(f'Rebalance decision @ {as_of_date.date()}\n')
    f.write(f'Strategy: U{len(MAIN_UNIVERSE)} + residual_mom + n={LONG_N} + rank + BTC {BTC_CAP*100:.0f}%\n\n')
    f.write(f'Main signal table:\n{sig_df.to_string()}\n')
    f.write(f'\nBTC signal: {btc_signal:+.4f}\n')
    f.write(f'BTC allocation this month: {btc_alloc*100:.1f}%\n')
    f.write(f'\nCurrent holdings: {current_holdings}\n')
    f.write(f'Cash balance: ${cash_balance:,.2f}\n')
    f.write(f'Total portfolio value: ${total_value:,.2f}\n')
    f.write(f'\nTarget weights:\n')
    for t, w in sorted(targets_pct.items(), key=lambda kv: -kv[1]):
        f.write(f'  {t}: {w*100:.2f}%  ({target_shares.get(t, 0)} shares)\n')
    if not trade_df.empty:
        f.write(f'\nTrades:\n{trade_df.to_string()}\n')

print(f'✅ 决策记录已保存到 {log_path}')